# Synthetic control : IRA & CHIPS Act et investissement non résidentiel

Question : à l'échelle du pays, l'investissement américain en bâtiments non résidentiels décroche-t-il de son jumeau synthétique après août 2022 ?
Nous suivons la méthode d'Abadie, Diamond et Hainmueller (2010), avec une inférence par permutation. Les calculs sont dans [`src/synth.py`](../src/synth.py).

In [1]:
import sys

import polars as pl

sys.path.append('../src')
import data_processing as dp
import synth
from plot_functions import set_theme, plot_ranking, plot_synth, plot_gaps, plot_specs

set_theme()
data = dp.load_processed()
OECD = data['oecd_gfcf_index2019_quarterly']

## 1. Design

Le jumeau synthétique est une moyenne pondérée des 20 autres pays de l'OCDE, $\hat{Y}_{USA,t} = \sum_j w_j\, Y_{j,t}$ avec $w_j \geq 0$ et $\sum_j w_j = 1$. Les poids sont choisis pour suivre les États-Unis au mieux de 2015-T1 à 2022-T2.
La méthode suppose que, sans les lois, les États-Unis auraient continué à suivre ce jumeau, que les donneurs ne sont pas touchés par les lois et qu'aucun autre choc propre aux États-Unis ne survient en 2022-T3.
Elle complète le DiD sur deux points : l'unité est le pays et non le secteur, et l'outcome est un volume déjà corrigé de l'inflation par l'OCDE. Cet outcome, la FBCF en bâtiments non résidentiels et ouvrages (base 100 = 2019), mélange toutefois bureaux, infrastructures et usines, si bien que l'effet attendu est dilué.

## 2. Poids

In [2]:
w = synth.weights(OECD).filter(pl.col('poids') > 0.001)
plot_ranking(w, 'pays', 'poids', n=w.height, title='Poids des donneurs dans le jumeau synthétique',
             subtitle='Poids non nuls, ajustés sur 2015-T1 → 2022-T2', xtitle='Poids')

Le jumeau repose sur 8 pays, dominés par l'Irlande (**0,27**), puis le Japon (0,15), la Suède, le Canada et la Finlande (0,13 chacun).
Le poids de l'Irlande pose problème car sa série est très volatile, et la section 6 teste le résultat sans elle.

## 3. Ajustement avant traitement

In [3]:
s = synth.synthetic(OECD)
plot_synth(s, title='États-Unis et jumeau synthétique',
           subtitle="FBCF en bâtiments non résidentiels et ouvrages, volume, base 100 = 2019")

In [4]:
pl.DataFrame([synth.summary(s)]).with_columns(pl.selectors.numeric().round(2))

rmspe_avant,rmspe_après,ratio,écart_moyen,écart_pic,trimestre_pic
f64,f64,f64,f64,f64,str
2.44,9.73,3.99,8.31,14.23,"""2024-T1"""


Le RMSPE avant traitement vaut **2,44** points d'indice. Le jumeau suit les États-Unis de près de 2015 à 2019, mais s'en écarte en 2020-2022, les États-Unis passant jusqu'à 6,6 points sous lui en 2021-T4.
L'ajustement est donc moyen, et l'écart positif observé après 2022 peut en partie refléter un rebond après ce creux.

## 4. Estimation

In [5]:
plot_gaps(s, title='Écart entre les États-Unis et leur jumeau synthétique', subtitle="Points d'indice (base 100 = 2019)")

Après 2022-T3, les États-Unis dépassent leur jumeau de **+8,3 points** en moyenne, avec un pic de +14,2 points en 2024-T1, puis l'écart redescend à +6,6 en 2026-T2.
Le RMSPE après traitement (9,73) vaut 3,99 fois le RMSPE avant.

## 5. Inférence

In [6]:
gaps, ratios = synth.space_placebos(OECD)
plot_gaps(gaps, title='Écarts des États-Unis et des placebos en espace',
          subtitle='Chaque pays traité à tour de rôle, États-Unis exclus des donneurs des placebos')

In [7]:
ratios.head(6).with_columns(pl.selectors.float().round(2))

rang,pays,rmspe_avant,rmspe_après,ratio
u32,str,f64,f64,f64
1,"""ITA""",4.08,53.53,13.13
2,"""FRA""",1.02,13.13,12.91
3,"""AUT""",1.79,17.81,9.96
4,"""ESP""",2.12,14.4,6.78
5,"""NLD""",1.39,8.36,6.03
6,"""USA""",2.44,9.73,3.99


Classés par ratio RMSPE après / avant, les États-Unis (3,99) n'arrivent qu'au **6e rang sur 21**, derrière l'Italie, la France, l'Autriche, l'Espagne et les Pays-Bas.
Plusieurs pays européens s'écartent donc davantage de leur propre jumeau après 2022.

In [8]:
synth.p_values(ratios).with_columns(pl.col('p').round(3))

placebos,pays,rang_usa,p
str,i64,i64,f64
"""Tous les pays""",21,6,0.286
"""RMSPE avant ≤ 2 × USA""",14,6,0.429


La p-valeur de rang vaut **0,286** (6/21), pour un minimum possible de 1/21 ≈ 0,048 avec 21 pays.
Si l'on retire les placebos mal ajustés (RMSPE avant supérieur à 2 fois celui des États-Unis), il reste 14 pays et p monte à 0,429. L'écart américain ne se distingue pas des placebos.

## 6. Robustesse

In [9]:
plot_specs(synth.robustness(OECD, data['oecd_donor_fit']), 'spécification', 'écart_moyen', 'Principale (2022-T3)',
           pct=False, title='Écart moyen selon la spécification',
           subtitle="Écart entre les États-Unis et leur jumeau après traitement, points d'indice", xtitle="Points d'indice")

Retirer un à un les autres donneurs, ou ne garder que les 5 mieux corrélés, laisse l'écart entre +7,7 et +9,2 points. Sans l'Irlande, il tombe à **+1,7**, si bien que le résultat repose sur un seul donneur.
Le placebo dans le temps (fausse date en 2019-T1) donne −2,2 points. Avancer le traitement à 2021-T3 réduit l'écart à +4,1, car la période après inclut alors le creux de 2021-2022.

## 7. Limites

| Menace | Risque | Effet sur l'estimation |
|---|---|---|
| Crise énergétique européenne | Près des deux tiers du poids vont à des pays européens (Irlande, Suède, Finlande, Espagne, Tchéquie), freinés en 2022-2023. | Jumeau trop bas, effet surestimé. |
| Spillover | L'IRA attire des projets qui auraient eu lieu en Europe ou au Canada. | Donneurs plus bas, effet surestimé. |
| Outcome trop large | Bureaux et infrastructures pèsent bien plus que les usines visées par les lois. | Effet dilué, donc sous-estimé. |
| Chocs propres aux États-Unis | Taux de la Fed plus élevés, rebond après le creux de 2021. | Sens ambigu : les taux freinent l'investissement, le rebond gonfle l'écart. |

## 8. À retenir

- **Effet pays** : +8,3 points en moyenne après 2022-T3 (pic de +14,2 en 2024-T1), bien en dessous de l'effet sectoriel du DiD (+189 % à +452 %), ce qui est cohérent avec un agrégat large.
- **Ajustement** : bon jusqu'en 2019, moyen en 2020-2022, avec des États-Unis sous leur jumeau juste avant les lois ; une partie de l'écart peut être un rebond.
- **Inférence** : 6e rang sur 21, p = 0,286 (0,429 sans les placebos mal ajustés) ; l'écart américain ne se distingue pas de ceux des autres pays.
- **Fragilité** : sans l'Irlande, l'écart tombe à +1,7 point.
- **Lien avec le DiD** : le synthetic control ne contredit pas l'effet fort sur le secteur traité, mais cet effet ne se voit pas clairement à l'échelle du pays.